# 에이전트 시스템 설계

## 첫 번째 에이전트 시스템

* 해결하려는 문제
    * 고객지원 팀은 매일 머그잔 환불 요청, 미배송 주문 취소, 배송지 변경 등을 묻는 이메일을 수십에서 수백 건 처리
    * 담당자는 다양한 작성된 메일을 읽고 백엔드에서 주문 내역을 조회해 적절한 API를 호출한 다음 확인 메일 작성  
    -> 반복적인 프로세스는 자동화에 매우 적합하지만 그 범위를 명확하게 정해야 함
* 특정 규칙과 가이드라인을 따르는 작업이라면 파운데이션 모델을 기반으로 시스템을 설계하는 정도로도 자동화가 가능함

* 만들 에이전트는 고객 메시지와 주문 기록을 받아 어떤 도구를 호출할지 결정하고 올바른 파라미터로 해당 도구를 실행한 다음 간단한 확인 메시지를 보냄
    * issue_refund
    * cancel_order
    * update_address_for_order

In [ ]:
from typing import TypedDict, Annotated, Sequence
import operator
from langchain.tools import tool
from langchain.chat_models import init_chat_model
from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage, ToolMessage
from langgraph.graph import StateGraph

# 환경변수 확인
import os
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass  

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError(
        "OPENAI_API_KEY가 설정되지 않았습니다."
        "환경변수 또는 .env 파일에서 설정해주세요."
    )


# State 타입 정의
class AgentState(TypedDict):
    order: dict
    messages: Annotated[Sequence[BaseMessage], operator.add]

# -- 1) 주문 취소 도구 정의
@tool
def cancel_order(order_id: str) -> str:
    """배송되지 않은 주문을 취소"""
    # 백엔드 API 호출
    return f"주문 {order_id}이(가) 취소되었습니다."

# -- 2) 에이전트 구조 정의: LLM 호출, 도구 실행, 다시 LLM 호출
def call_model(state):
    msgs = state["messages"]
    order = state.get("order", {"order_id": "UNKNOWN"})

    # LLM 초기화
    llm = init_chat_model(model="gpt-5-mini", temperature=0)
    llm_with_tools = llm.bind_tools([cancel_order]) # 도구 바인딩

    # 시스템 프롬프트에 모델이 할 일을 정확히 명시
    prompt = (
        f'''
        당신은 이커머스 지원 에이전트입니다.
        주문ID: {order['order_id']}
        고객이 취소를 요청하면 cancel_order(order_id)를 호출하고
        간단한 확인 메시지를 보내세요.
        그렇지 않으면 일반적으로 응답하세요.
        '''
    )
    full = [SystemMessage(content=prompt)] + msgs

    # 1차 LLM 패스: 도구 호출 여부 결정
    first = llm_with_tools.invoke(full)
    out = [first]

    if getattr(first, "tool_calls", None):
        # cancel_order 도구 실행
        tc = first.tool_calls[0]
        result = cancel_order.invoke(tc["args"])
        out.append(ToolMessage(content=result, tool_call_id=tc["id"]))

        # 2차 LLM 패스: 최종 확인 텍스트 생성
        second = llm.invoke(full + out)
        out.append(second)

    return {"messages": out}

# -- 3) StateGraph로 에이전트 구조 연결
def construct_graph():
    g = StateGraph(AgentState) # TypedDict 사용
    g.add_node("assistant", call_model)
    g.set_entry_point("assistant")
    return g.compile()

graph = construct_graph()

if __name__ == "__main__":
    example_order = {"order_id": "B73973"}
    convo = [HumanMessage(content="주문 #B73973를 취소해주세요.")]
    result = graph.invoke({"order": example_order, "messages": convo})
    for msg in result["messages"]:
        print(f"{msg.type}: {msg.content}")

* 에이전트의 작업 범위를 지나치게 좁히면 자주 생기는 다른 요청을 놓쳐 큰 효과를 보지 못함
* 범위를 너무 넓히면 에이전트가 수많은 엣지케이스를 대응하도록 준비해야 해서 작업 기간이 오래 걸림

* 에이전트에 대해 다음 사항을 위주로 평가
    * 올바른 도구(cancel_order)를 호출했는가?
    * 올바른 파라미터(정확한 주문ID)를 전달했는가?
    * 고객에게 명확하고 정확한 확인 메시지를 보냈는가?

* 에이전트를 직접 테스트하는 간단한 평가 예시

In [ ]:
from langchain_core.messages import HumanMessage, ToolMessage

# 최소 평가
example_order = {"order_id": "B73973"}
convo = [HumanMessage(content="""더 저렴한 곳을 찾았습니다.
                                주문 #B73973을 취소해 주세요.""")]
result = graph.invoke({"order": example_order, "messages": convo})

# 도구 호출 확인: tool_calls 속성 또는 ToolMessage 타입 확인
has_tool_call = any(
    getattr(m, "tool_calls", None) or isinstance(m, ToolMessage)
    for m in result["messages"]
)
assert has_tool_call

# 취소 확인 메시지 확인
assert any("취소" in str(m.content) for m in result["messages"]), "확인 메시지가 누락됨"

print("에이전트가 최소 평가 기준을 통과했습니다!")

* 도구가 호출되었고 확인 메시지가 전송되었는지 확인함
* 실제 평가는 더욱 깊게 들어감
* @tool 데코레이터로 자동화했기 때문에 실제 티켓을 대상으로 테스트를 작성하기 매우 쉬워짐
    * 도구 재현율, 파라미터 정확도, 확인 품질 같은 측정 지표를 곧바로 얻음

## 에이전트 시스템의 핵심 구성요소

* 효과적인 에이전트 기반 시스템을 설계하려면 에이전트가 임무를 잘 수행하도록 돕는 핵심 구성요소를 깊이 이해해야 함
    * 각 구성요소는 에이전트의 능력, 효율, 적응성을 결정함
* 동적인 복잡한 환경에서 에이전트가 제대로 운영되려면 모델 선택부터 도구, 메모리, 계획 능력 부여에 이르기까지 많은 요소가 함께 작동해야 함

## 모델 선택

* 모든 에이전트 기반 시스템의 중심에는 모델이 있음
* 모델은 에이전트의 의사결정, 상호작용, 학습 역량을 결정  
-> 에이전트 시스템 구축은 모델을 선택하는 데서 시작함
* 모델은 에이전트가 입력을 해석하고 출력을 생성하며 환경에 적응하는 바익을 결정하므로 모델 선택은 시스템의 성능, 확장성, 지연시간, 비용에 직접적인 영향을 미침
* 적절한 모델 선택은 작업의 복잡성, 입력 데이터의 특성, 인프라 제약, 일반성-속도-정밀성의 트레이드오프에 달려 있음

* 모델 선택은 작업 복잡도 평가에서 시작함
* 대형 파운데이션 모델은 개방형 환경에서 컨텍스트 이해, 유연한 추론, 창의적 생성이 필요한 에이전트에 적합함
    * 일반화가 뛰어나고 모호성, 문맥적 뉘앙스, 다단계 작업에 강함
    * 다만 높은 연산 자원, 클라우드 인프라, 더 큰 지연시간을 요구
* 소형 모델은 정의가 명확한 반복 작업에 적합한 경우가 많음
    * 로컬 하드웨어에서 효율적으로 실행되고 응답이 빠르며 배포 및 운영 비용이 낮음

* 최근에는 모달리티가 중요해짐
* 텍스트에 더불어 이미지, 오디오, 구조화 데이터까지 처리해야 하는 경우가 늘어남
* 멀티모달 모델은 다양한 입력을 결합해 해석
* 텍스트 중심은 텍스트 전용 모델이 단순성과 추론 속도 측면에서 유리

* 개방성과 커스터마이징 가능성도 중요한 고려 사항
* 오픈 소스 모델은 투명하고 필요에 따라 파인튜닝, 수정이 가능함  
-> 규제 환경이나 프라이버시가 중요한 도메인에서 특히 유리함

* 범용 사전학습 모델을 사용할지, 맞춤형 모델을 준비할지는 도메인의 특수성과 민감도에 달림

* 실제 배포에서는 비용, 지연시간이 모델 선택의 결정적 요인이 되곤 함
* 가능하면 자신의 작업을 기준으로 직접 모델을 비교해 가성비가 가장 좋은 모델을 고르는 편이 좋음

* 모델 선택은 한 번 결정하고 끝날 문제가 아니라 에이전트 역량, 사용자 요구, 인프라가 진화함에 따라 지속적으로 재검토해야 함
    * 일반성 vs 전문성
    * 성능 vs 비용
    * 단순성 vs 확장성
* 작업 복잡도, 입력 모달리티, 운영 제약, 커스터마이징을 신중히 고려해 효율적으로 행동하고 안정적으로 확장하며 정밀하게 수행하는 모델을 선택해야 함

## 도구

* 도구는 에이전트가 특정 작업을 수행하고 문제를 해결할 수 있게 함
* 도구는 에이전트의 기능적 빌딩 블록으로 작업 실행과 사용자 및 다른 시스템과의 상호작용을 가능하게 함

### 특정 작업을 해결하는 도구 설계

* 도구는 보통 에이전트가 해결하려는 작업에 맞춰 설계
* 설계할 때는 다양한 조건과 컨텍스트에서 에이전트가 어떻게 작업을 수행할지 고려해야 함

* 로컬 도구
    * 에이전트가 외부 의존성 없이 내부 로직과 계산만으로 작업을 수행
    * 주로 규칙 기반이나 사전에 정의된 함수를 실행하는 방식
    * 수학 계산, 로컬 데이터베이스 조회, 미리 정한 규칙에 따른 간단한 의사결정 등이 해당

* API 기반 도구
    * 에이전트가 외부 서비스나 데이터 소스와 상호작용하도록 함
    * 실시간 데이터를조회하거나 외부 시스템을 활용해 로컬 환경 너머의 기능을 수행할 수 있음

* MCP(Model Context Protocol)
    * 표준 스키마를 통해 사용자 프로필, 대화 이력, 세계 상태, 작업 메타데이터 같은 구조화된 실시간 컨텍스트를 프롬프트에 직접 전달
    * 외부 서비스를 호출하고 응답을 기다려야 하는 기존 API 기반 도구와는 달리 MCP는 표준 스키마를 통해 구조화된 실시간 컨텍스트를 제공하고 도구 호출을 표준화된 방식으로 지원
    * 불필요한 도구 사용르 줄이고 대화 상태를 보존하며 실시간 상황 인식을 모델에 주입하는 데 특히 효과적

### 도구 통합과 모듈성

* 도구 개발에서 모듈형 설계는 필수
* 각 도구는 필요에 따라 쉽게 통합하거나 교체할 수 있도록 독립적인 모듈로 설계해야 함
    * 전체 시스템을 다시 구축하지 않고도 에이전트의 기능을 확장하거나 업데이트할 수 있음

## 메모리

* 메모리는 에이전트가 정보를 저장하고 검색할 수 있게 하는 핵심 구성 요소
* 이를 통해 에이전트는 컨텍스트를 유지하고 과거 상호작용에서 학습하며 시간이 지남에 따라 더 나은 의사결정을 내릴 수 있음
* 효과적인 메모리 관리가 이루어지면 에이전트는 변화하는 환경에서도 효율적으로 작동하고 과거 데이터를 기반으로 새로운 상황에 적응할 수 있음

### 단기 메모리

* 에이전트가 현재 작업이나 대화와 관련된 정보를 저장하고 관리하는 능력
    * 상호작용 중 컨텍스트를 유지하는 데 사용
    * 에이전트가 실시간으로 일관된 의사결정을 내릴 수 있도록 도움

* 단기 메모리는 종종 롤링 컨텍스트 윈도 방식을 통해 구현
    * 최근 정보를 일정 범위 내에서 계속 갱신하면서 오래된 데이터를 버릴 수 있게 함
    * 챗봇이나 가상 비서 등에서 유용함

### 장기 메모리

* 에이전트가 오랜 기간에 걸쳐 지식과 경험을 저장하도록 하여 과거 정보를 바탕으로 향후 행동을 결정할 수 있게 함
    * 시간에 따라 점차 발전해야 하거나 사용자 선호도에 기반한 개인화된 경험을 제공해야 하는 에이전트에 중요함

* 주로 데이터베이스, 지식 그래프 또는 파인튜닝된 모델을 통해 구현
    * 에이전트가 사용자 선호도나 과거 성능 지표 같은 구조화된 데이터를 저장하고 필요할 때 검색할 수 있게 함

### 메모리 관리 및 검색

* 효과적인 메모리 관리는 저장된 데이터를 체계적으로 구성하고 인덱싱해 필요 시 쉽게 검색할 수 있도록 하는 것을 의미
* 관련 있는 데이터와 없는 데이터를 구분하고 필요한 정보를 신속하게 찾아야 함
* 일부 정보를 잊어야 할 수도 있음

## 오케스트레이션

* 개별적인 기능을 엔드투엔드 솔루션으로 전환
* 여러 스킬을 배치하고 상황에 따라 실행하며 전체 과정을 감독하는 논리로 각 단계가 다음 단계로 자연스럽게 이어지며 명확한 목표를 이루도록 함
* 핵심은 도구나 스킬 호출의 가능한 순서를 평가하고 그 결과를 예측하며 여러 단계를 거치는 작업에서 가장 성공 가능성이 높은 경로를 선택하는 것에 있음

* 오케스트레이터는 진행 상황과 환경을 지속적으로 모니터링하면서 필요에 따라 워크플로를 일시 중단하거나 경로를 재조정해 목표에서 벗어나지 않도록 해야함
* 많은 상황에서 에이전트는 점진적으로 계획을 구축
    * 몇 단계를 먼저 실행한 뒤 최신 결과를 바탕으로 남은 워크플로를 다시 평가하고 업데이트

## 설계 트레이드오프

* 에이전트 기반 시스템을 설계할 때는 성능, 확장성, 신뢰성, 비용 사이의 다양한 트레이드오프를 균형 있게 조정해야 함

### 성능: 속도와 정확도의 균형

* 높은 성능은 정보를 빠르게 처리하고 의사결정을 내리며 작업을 수행할 수 있지만 그만큼 정밀도가 떨어질 수 있음
* 정확도를 높이려면 전체 속도가 느려질 수 있음

* 하이브리드 전략
    * 에이전트가 먼저 빠르고 대략적인 결과를 제시한 뒤 추가로 시간과 데이터를 활용해 이를 정교하게 보완하는 방식
    * 추천 시스템이나 진단 시스템에서 흔히 사용

### 확장성: 에이전트 시스템의 엔지니어링적 확장

* 현대의 에이전트 시스템은 딥러닝 모델과 실시간 처리에 크게 의존  
-> 확장성은 중요한 기술적 과제
* 시스템이 커질수록 데이터량, 동시 처리 작업 수, 연산 리소스의 효율적인 관리가 핵심

* GPU는 에이전트 시스템 확장에서 가장 비싸고 제약이 큼
* 올바른 리소스 관리를 통해 에이전트는 고성능 연산에 따른 지연시간과 비용을 최소화하면서 늘어나는 워크로드를 안정적으로 처리할 수 있음

* GPU 동적 할당
    * 실시간 수요에 따라 GPU를 배정해 유휴 시간을 줄이고 활용도를 극대화

* 탄력적 GPU 프로비저닝
    * 클라우드나 온프레미스 GPU 클러스터를 이용해 현재 워크로드에 따라 자동으로 리소스를 확장하거나 축소

* 우선순위 큐잉과 지능형 작업 스케줄링을 결합하면 중요한 작업은 즉시 GPU 접근 권한을 부여하고 덜 중요한 작업은 대기시켜 효율성을 높일 수 있음

* 지연시간이 문제
* 에이전트가 실시간 또는 준실시간 환경에서 상호작용해야 하는 경우 최소 지연으로 작동할 수 있도록 스케줄링 해야 함

* 비동기 작업 실행

* 동적 로드 밸런싱
    * 활용률이 낮은 GPU로 작업을 분산시켜 특정 GPU에 병목이 생기지 않도록 함

* 수평 확장
    * 시스템의 처리 능력을 높이기 위해 GPU 노드를 추가하는 방식

* 워크로드가 유동적인 시스템의 경우 하이브리드 클라우드 방식 활용 시 확장성을 높일 수 있음
* 온프레미스 GPU 리소스와 클라우드 기반 GPU를 결합해 요청이 몰릴 경우 버스트 스케일링으로 클라우드 GPU에 일시적으로 작업을 분산시켜 처리 용량을 확장하고 수요가 줄어들면 리소스를 해제

### 신뢰성: 견고하고 일관된 에이전트

* 신뢰성은 에이전트가 일정한 기간동안 일관되고 정확하게 작업을 수행할 수 있는 능력
* 예상된 상황과 예기치 못한 조건에서도 문제없이 작동해야 함

#### 장애 허용

* 신뢰성의 한 가지 핵심 요소는 에이전트가 오류나 예기치 못한 사건을 감지하고 비정상 종료나 예측 불가능한 오작동 없이 처리할 수 있도록 하는 것  
-> 장애 허용 메커니즘 구축이 피룡함

* 장애 허용 시스템은 문제를 감지하고 정상 상태로 복구할 수 있어야 함
* 일반적으로 중복 구조를 사용

#### 일관성과 견고성

* 신뢰성을 확보하려면 다양한 시나리오, 입력, 환경에서 일관된 성능을 유지해야 함
* 이상적인 조건뿐 아니라 다양한 엣지 케이스, 스트레스 테스트, 현실적 제약 조건에서도 안정적으로 작동하도록 해야 함

* 신뢰성을 달성하기 위한 접근
    * 철저한 테스트
        * 단위 테스트, 통합 테스트, 실제 환경을 시뮬레이션한 테스트 등 다양한 검증 과정을 거쳐야 함
        * 테스트는 엣지 케이스, 예상치 못한 입력, 적대적 조건 등을 포괄해야 함
    * 모니터링과 피드백 루프
        * 지속적인 모니터링을 통해 이상 징후를 탐지하고 환경 변화에 맞춰 작동 방식을 조정해야 함
        * 피드백 루프 활용 시 에이전트가 환경으로부터 학습하고 성능을 점진적으로 향상시킬 수 있음

### 비용: 성능과 지출의 균형

* 비용은 매우 중요한 고려 요소
* 에이전트를 개발, 배포, 유지하는 데 드는 비용은 기대되는 이익과 투자 수익률(ROI)을 기준으로 평가해야 함
* 비용은 모델 복잡도, 인프라, 확장성과 관련된 의사결정 전반에 영향을 미침

#### 개발 비용

* 고도화된 에이전트를 개발하는 과정은 비용이 많이 필요함
    * 특히 대규모 데이터셋, 전문 인력, 막대한 연산 자원이 필요한 머신러닝 모델을 사용하는 경우
    * 여기에 반복적인 설계, 테스트, 최적화의 필요로 개발 비용은 더욱 증가

* 복잡한 에이전트를 구축하려면 데이터 과학자, 머신러닝 엔지니어, 도메인 전문가 등으로 구성된 팀이 필요함
* 신뢰성, 확장성을 갖춘 에이전트 시스템을 구축하려면 방대한 테스트 인프라가 필요함  
-> 시뮬레이션 환경을 마련하고 테스트 도구와 프레임워크에 투자해 견고한 기능을 검증해야 함

#### 운영 비용

* 배포 이후 에이전트를 실행하는 데 운영 비용이 상당해질 수 있음
    * 특히 실시간 의사결정이나 지속적인 데이터 처리가 필요한 시스템의 경우 고성능 연산이 필수적
* 방대한 데이터를 처리하거 대규모 메모리를 유지하는 에이전트는 데이터 저장 및 대역폭 비용이 크게 잡아먹음
* 버그 수정, 시스템 개선 등 정기적인 유지보수, 업데이트가 더해지면 장기적으로 시스템의 성능, 신뢰성을 보장하기 위한 인적, 기술적 자원 투입이 불가피

#### 비용 대비 가치

* 에이전트 시스템의 비용은 그 가치로 정당화되어야 함
* 비용에 대한 모든 의사결정은 시스템의 전체 목표와 예상 수명을 고려해 이루어져야 함

* 비용을 최적화하는 대표적인 전략
    * 경량 모델
        * 특정 작업에서 규칙 기반 시스템이 딥러닝 모델과 유사한 성능을 낼 수 있으면 단순한 접근이 훨씬 비용 효율적
        * 필요에 따라 더 간단하고 효율적인 모델을 사용하면 개발 및 운영 비용을 모두 절감할 수 있음
    * 클라우드 기반 리소스
        * 클라우드 컴퓨팅 자원을 활용하면 초기 인프라 구축 비용을 줄이고 사용량 기반 과금 모델을 통해 확장성과 유연성을 확보할 수 있음
    * 오픈소스 모델과 도구
        * 오픈소스 머신러닝 라이브러리와 프레임워크 활용 시 소프트웨어 개발 비용을 줄이면서도 높은 품질의 에이전트 구현 가능

* 비용은 개발, 운영 두 측면에서 모두 고려해 예산 범위 내에서 최대 가치를 제공하는 시스템을 구축해야 함

## 아키텍처 디자인 패턴

* 에이전트 기반 시스템의 아키텍처 설계는 에이전트가 어떤 구조로 이루어져 있으며 환경과 어떻게 상호작용하고 어떤 방식으로 작업을 수행하는지를 결정
* 아키텍처 선택은 확장성, 유지보수성, 유연성에 직접적인 영향을 미침

### 단일 에이전트 아키텍처

* 가장 단순하고 직관적인 설계 형태로 하나의 에이전트가 시스템 내 모든 작업을 관리하고 실행
* 스스로 의사결정, 계획 수립, 실행을 수행

* 명확히 정의된 좁은 범위의 작업에 적합
* 하나의 엔티티로 처리할 수 있는 업무에 이상적
* 구조가 단순해 설계, 개발, 배포가 용이

* 문제 도메인이 명확하고 작업이 단순하며 대규모 확장이 필요하지 않은 환경에서 효과적
* 고객 지원 챗봇, 범용 어시스턴트, 코드 생성 에이전트 등

### 멀티 에이전트 아키텍처

* 여러 에이전트가 공동의 목표를 달성하기 위해 협력
* 각 에이전트는 작업의 성격에 따라 독립적으로, 병렬적으로, 혹은 조율된 방식으로 작동할 수 있음
* 복잡한 환경에서 유용함
    * 작업의 여러 측면을 전문화된 에이전트가 분담하거나 병렬 처리를 통해 효율성과 확장성을 높일 수 있는 상황에 적합함

* 멀티 에이전트 시스템의 장점
    * 협업과 전문화
        * 각 에이전트는 특정 작업이나 영역에 특화되도록 설계 가능
    * 병렬성
        * 여러 작업을 동시에 수행할 수 있음
    * 확장성 향상
        * 새로운 에이전트를 추가해 더 많은 작업을 처리하거나 부하를 분산할 수 있음
    * 중복성과 복원력
        * 여러 에이전트가 독립적으로 작동하므로 일부가 작업에 실패해도 전체 시스템이 중단되지 않음
        * 다른 에이전트가 실패한 역할을 대신 수행할 수도 있어 신뢰성 향상

* 멀티 에이전트 시스템의 어려움
    * 조율과 통신의 복잡성
        * 에이전트 간 통신을 효율적으로 관리해야 함
        * 각 에이전트는 중복 작업, 충돌, 자원 경쟁을 피하기 위해 정보를 교환하고 작동 방식을 조율해야 함
    * 복잡성 증가
        * 설계, 개발, 유지보수가 훨씬 어려움
        * 통신 프로토콜, 조율 전략, 동기화 메커니즘을 추가로 고려해야 함
    * 효율성 저하
        * 토큰 소비가 많아 효율이 낮아질 수 있음
        * 빈번히 통신하고 컨텍스트를 공유하며 행동을 조율해야 하므로 처리 자원과 계산 비용이 더 많이 듦

* 멀티 에이전트 아키텍처는 작업이 복잡하거나 분산되어 있거나 구성 요소별 전문화가 필요한 환경에 잘 맞음

## 모범 사례

* 에이전트 기반 시스템은 알맞은 모델, 스킬, 아키텍처만으로 완성되지 않음
* 실제 환경에서 최적으로 작동하고 환경 변화에 맞춰 지속적으로 발전하려면 개발 라이프사이클 전반에 걸쳐 모범 사례를 따라야 함

### 점진적 설계

* 에이전트 개발의 핵심 접근법
* 작은 프로토타입을 만들고 피드백을 반영해 여러 차례에 걸쳐 개선

* 주요 이점
    * 문제의 조기 발견
        * 초기 프로토타입을 배포하면 설계 결함, 성능 병목을 조기에 찾을 수 있음
        * 빠르게 문제를 수정해 장기 개발 비용을 줄이고 대규모 리팩터링을 피함
    * 사용자 중심 설계
        * 점진적 설계는 이해관계자와 최종 사용자, 개발자에서 빈번한 피드백을 받을 수 있음
        * 실환경에서 에이전트를 시험하며 행동과 응답을 세밀하게 조정해 사용자 요구와 기대에 잘 맞출 수 있음
    * 확장성
        * 최소 기능 제품(MVP)나 기초 에이전트로 시작하면 관리 가능한 단위로 점진적 성장 진행 가능
        * 시스템이 성숙해질수록 기능과 역량을 단계적으로 추가해 배포 전 충분히 검증 가능

* 점진적 설계 원칙
    * 프로토타입을 빠르게 개발
    * 테스트하고 피드백을 수집
    * 개선하고 반복

* 효과적인 점진적 설계는 작동하는 프로토타입을 빠르게 만들고 각 반복마다 피드백을 수집하며 얻은 인사이트를 바탕으로 시스템을 지속적으로 다듬어 성능과 사용성 목표를 달성

### 평가 전략

* 에이전트 기반 시스템의 성능과 신뢰성 평가는 개발 과정에서 핵심

* 견고한 평가 프로세스는 에이전트 기능의 모든 측면을 포괄하는 종합 테스트 프레임워크를 수립하는 일로 시작
* 예상된 시나리오와 예상치 못한 시나리오 모두에서 에이전트를 철저히 검증

* 기능 테스트는 에이전트가 핵심 작업을 수행하는지 확인하는 데 초점
* 주요 초점 영역
    * 정확성
        * 설계에 기반해 에이전트가 일관되게 정확하고 기대한 출력을 제공하는지 확인
    * 경계 테스트
        * 매우 큰 데이터셋, 이례적 질의, 모호한 지침과 같은 엣지 케이스를 에이전트가 어떻게 처리하는지 평가
    * 작업별 지표
        * 도메인별 작업을 처리하는 에이전트의 경우 해당 도메인의 정확도와 컴플라이언스 요구사항을 충족하는지 확인

* 에이전트는 종종 원래 학습 도메인 밖의 작업을 접함
* 견고한 평가는 광범위한 재학습 없이도 새로운 작업에 적응하는 능력을 검증해야 함
    * 범용 에이전트, 동적인 환경에서 작동하도록 설계된 에이전트에 특히 중요함

* 사용자 경험은 에이전트 시스템의 성공을 좌우하는 핵심 요소
* 실제 환경에서 사용자 기대를 얼마나 충족하는지도 평가해야 함

* 실제 사용자로부터 피드백을 수집하면 에이전트의 작동 방식을 개선해 효과성과 사용자 만족도를 높이는 데 도움이 됨
    * 사용자 만족도 점수
        * NPS나 고객만족도(CSAT) 같은 지표로 만족도 측정
    * 작업 완료율
        * 사용자가 에이전트의 도움으로 작업을 성공적으로 완료하는 비율을 측정
    * 명시적 신호
        * 좋아요, 싫어요, 별점, 생성 결과의 수락, 거절, 수정 등 사용자가 피드백을 제공할 기회를 만듦
    * 암묵적 신호
        * 사용자-에이전트 상호작용을 분석해 오해, 지연, 감정, 부적절한 응답 등 공통 문제 지점을 식별

* 인간 개입 검증은 자동화된 평가와 인간 판단을 결합해 에이전트 성능이 실제 표준과 일치하도록 함
* 전문가가 에이전트의 출력의 샘플을 검토해 정확성, 윤리 준수, 모범 사례와의 정렬을 확인하고 검토 결과를 자동화된 평가의 보정과 개선에 활용

* 평가는 실제 적용 환경을 가깝게 모사해 수행해야 함  
-> 이를 통해 시스템이 통제된 개발 환경 밖에서도 신뢰성 있게 작동함을 보장
* 엔드투엔드 테스트는 여러 시스템, 데이터 소스, 플랫폼 전반에서 에이전트가 기대한 대로 작동하는지 확인

### 실환경 테스트

* 실환경 테스트는 에이전트가 라이브 환경의 예측 불가능성과 복잡성을 처리할 수 있도록 보장하는 데 필수

* 실환경의 복잡성 노출
    * 통제된 환경에서는 에이전트가 예측 가능한 입력과 응답으로 작동  
    -> 실제 환경은 예측하기 어려움
    * 실제 시나리오의 복잡성과 변동성을 처리할 수 있는지 확인할 수 있음
* 엣지 케이스 발굴
    * 실환경 상호작용은 설계나 테스트 단계에서 고려하지 못했던 엣지 케이스를 자주 드러냄
* 부하 상태에서의 성능 평가
    * 실환경 테스트를 통해 높은 작업량이나 사용자 수요 증가 상황에서 에이전트가 어떻게 작동하는지 관찰할 수 있음

* 실환경 테스트는 현실 조건에서의 성능을 검증해 에이전트의 배포 준비 상태를 보장

* 단계적 배포
    * 제한된 환경에서의 소규모 테스트로 시작해 배포 준비 상태를 보장
    * 과부하 없이 문제를 점진적으로 식별해 해결
* 에이전트 작동 방식 모니터링
    * 모니터링 도구로 실환경 테스트 동안 에이전트의 작동 방식, 응답, 성능 지표를 추적
    * KPI에 집중해야 함
* 사용자 피드백 수집
    * 사용자와 소통해 상호작용 경험에 대한 피드백을 수집
* 인사이트에 따른 반복
    * 실환경 테스트에서 얻은 인사이트를 개발 주기로 되돌림
    * 에이전트를 정교화하고 능력을 향상하고 향후 반복에서 성능을 최적화함

* 적응 가능하고 확장 가능하며 회복력 있는 에이전트 기반 시스템을 구축하는 데는 점진적 설계, 애자일 개발, 실환경 테스트 등의 모범 사례를 따르는 것이 중요함